# 09 — PyTorch and CNN Architectures

**CS231n Lectures 8–9.** You have now written every piece of a convnet by hand.
That was the point: from here on you use a framework, and you should use it
knowing exactly what it is doing for you.

This notebook covers the framework, then the five architectures worth knowing,
then trains a real ResNet on CIFAR-10 — on CPU, in about two minutes.

| You write | |
|---|---|
| `conv_params`, `conv_flops` | where the parameters and the compute actually go |
| `receptive_field` | how much of the image one deep unit sees |
| `count_parameters` | |
| `ResidualBlock.forward` | the one idea that made depth work |

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import torch
import torch.nn as nn
import torch.nn.functional as F

from cs231n_utils import (get_cifar10, CLASSES, rel_error, grade, show_solution,
                          use, show_images, plot_series, setup_plots, SERIES)

setup_plots()
torch.manual_seed(0)
np.random.seed(0)

print("torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", device, "| CPU threads:", torch.get_num_threads())

## 1. The three levels of abstraction

PyTorch is three APIs stacked on each other. Knowing which level you are at
prevents most confusion.

| Level | What you manage | When |
|---|---|---|
| **Tensors + autograd** | everything by hand | research, custom ops |
| **`nn.Module`** | layers; autograd handles gradients | the normal case |
| **`nn.Sequential` + optim** | almost nothing | straight-line models |

### Level 1 — tensors and autograd

The whole of notebook 04, done for you. `requires_grad=True` tells PyTorch to
build the computational graph as you compute; `.backward()` walks it.

In [ ]:
# The staged-computation example from notebook 04, now with autograd.
w0 = torch.tensor(2.0, requires_grad=True)
w1 = torch.tensor(-3.0, requires_grad=True)
w2 = torch.tensor(-3.0, requires_grad=True)
x0 = torch.tensor(-1.0)
x1 = torch.tensor(-2.0)

f = torch.sigmoid(w0 * x0 + w1 * x1 + w2)
f.backward()

print("f = %.6f" % f.item())
print("autograd:  dw0 %.5f  dw1 %.5f  dw2 %.5f" % (w0.grad, w1.grad, w2.grad))
print("notebook 04 got: dw0 -0.19661  dw1 -0.39322  dw2 0.19661")

### Your numpy conv backward vs autograd

A worthwhile check: does the `conv_backward_naive` you wrote in notebook 06
agree with PyTorch's? If it does, you understand the real thing.

In [ ]:
from solutions.nb06 import conv_forward_naive, conv_backward_naive

rng = np.random.RandomState(0)
xn = rng.randn(2, 3, 6, 6)
wn = rng.randn(4, 3, 3, 3)
bn_ = rng.randn(4)
cp = {'stride': 1, 'pad': 1}

out_np, cache = conv_forward_naive(xn, wn, bn_, cp)
dout = rng.randn(*out_np.shape)
dx_np, dw_np, db_np = conv_backward_naive(dout, cache)

xt = torch.tensor(xn, requires_grad=True)
wt = torch.tensor(wn, requires_grad=True)
bt = torch.tensor(bn_, requires_grad=True)
out_t = F.conv2d(xt, wt, bt, stride=1, padding=1)
out_t.backward(torch.tensor(dout))

print("forward  rel error %.2e" % rel_error(out_np, out_t.detach().numpy()))
print("dx       rel error %.2e" % rel_error(dx_np, xt.grad.numpy()))
print("dw       rel error %.2e" % rel_error(dw_np, wt.grad.numpy()))
print("db       rel error %.2e" % rel_error(db_np, bt.grad.numpy()))

Machine precision. Your four nested loops and cuDNN compute the same function
— the difference is entirely engineering.

### Level 2 and 3 — modules

Two rules that cause most beginner bugs:

- **`optimizer.zero_grad()` every step.** PyTorch *accumulates* gradients into
  `.grad` rather than overwriting — deliberately, so you can split a batch
  across several backward passes. Forget it and your gradients are the running
  sum of every step so far.
- **`model.train()` / `model.eval()`.** This switches BatchNorm between batch
  and running statistics, and turns dropout on and off. Exactly the train/test
  distinction from notebooks 07 and 08.

In [ ]:
model = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(16 * 16 * 16, 10))

xb = torch.randn(8, 3, 32, 32)
yb = torch.randint(0, 10, (8,))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
lossf = nn.CrossEntropyLoss()

model.train()
for step in range(3):
    opt.zero_grad()                      # <- the one everyone forgets
    loss = lossf(model(xb), yb)
    loss.backward()
    opt.step()
    print("step %d  loss %.4f" % (step, loss.item()))

In [ ]:
# What happens if you forget zero_grad: gradients pile up.
m2 = nn.Linear(4, 1)
x2, y2 = torch.randn(5, 4), torch.randn(5, 1)
norms = []
for step in range(4):
    loss = ((m2(x2) - y2) ** 2).mean()
    loss.backward()                      # no zero_grad, no step
    norms.append(m2.weight.grad.norm().item())
print("gradient norm after each backward (no zero_grad):")
print("  " + "  ".join("%.4f" % n for n in norms))
print("-> it is 1x, 2x, 3x, 4x the true gradient. Silent, and wrong.")

> **`nn.CrossEntropyLoss` takes raw scores, not probabilities.** It applies
> log-softmax internally, for the numerical-stability reason from notebook 05.
> Putting a `nn.Softmax` before it is a real and common bug: the model still
> trains, just badly.

## 2. Where parameters and compute actually go

Two different quantities that people conflate:

$$ \text{params} = C_{out}\,C_{in}\,k^2 + C_{out} \qquad
   \text{FLOPs} \approx C_{out}\,C_{in}\,k^2 \cdot H_{out} W_{out} $$

The parameter count does **not** depend on spatial size. The FLOP count does.
The consequence is a rule of thumb worth knowing:

> **Early layers dominate compute** (large spatial maps, few channels).
> **Late layers dominate parameters** (small maps, many channels).

### TODO 1 — the two formulas

In [ ]:
def conv_params(in_ch, out_ch, k, bias=True):
    """Number of learnable scalars in a Conv2d."""
    pass  # TODO

def conv_flops(in_ch, out_ch, k, out_h, out_w):
    """Multiply-accumulates for one forward pass on ONE image."""
    pass  # TODO

In [ ]:
grade('nb09', 'conv_params', conv_params, verbose=False)
grade('nb09', 'conv_flops', conv_flops, verbose=False)

In [ ]:
cpar = use('nb09', 'conv_params', conv_params)
cfl  = use('nb09', 'conv_flops', conv_flops)

# A VGG-16-like stack on 224x224, to see the split.
spec = [
    ('conv1  3->64',    3,   64,  3, 224),
    ('conv2  64->128',  64,  128, 3, 112),
    ('conv3  128->256', 128, 256, 3, 56),
    ('conv4  256->512', 256, 512, 3, 28),
    ('conv5  512->512', 512, 512, 3, 14),
]
rows = []
for nm, ci, co, k, hw in spec:
    rows.append((nm, cpar(ci, co, k), cfl(ci, co, k, hw, hw)))
tot_p = sum(r[1] for r in rows)
tot_f = sum(r[2] for r in rows)

print("%-18s %12s %6s %15s %6s" % ("layer", "params", "%", "MACs", "%"))
for nm, p, f in rows:
    print("{:<18} {:>12,} {:>5.1f}% {:>15,} {:>5.1f}%".format(
        nm, p, 100 * p / tot_p, f, 100 * f / tot_f))
print("{:<18} {:>12,} {:>6} {:>15,}".format("TOTAL", tot_p, "", tot_f))

# and the fully-connected head VGG used to have
fc1 = 512 * 7 * 7 * 4096 + 4096
print("\nVGG's first FC layer alone: {:,} parameters".format(fc1))
print("that is %.1fx all five conv stacks above combined" % (fc1 / tot_p))

That last number is why **global average pooling** replaced the giant
fully-connected head. GoogLeNet and every ResNet average each final feature map
to a single number, so the classifier is a tiny `Linear(512, num_classes)`
instead of 102 million parameters. Same accuracy, a fraction of the size, and
it accepts any input resolution.

### TODO 2 — receptive field

How much of the *input image* influences one unit deep in the network?

$$ r_{out} = r_{in} + (k - 1)\cdot j_{in} \qquad j_{out} = j_{in}\cdot s $$

where `j` ("jump") is the spacing in input pixels between adjacent units at
that layer, starting at `r = 1, j = 1`.

This is the argument for **stacking small filters**. Three 3×3 convs have the
same 7×7 receptive field as one 7×7 conv, but use `3·(3²C²) = 27C²` parameters
instead of `49C²`, and insert two extra nonlinearities. That observation is
essentially the whole VGG paper.

In [ ]:
def receptive_field(specs):
    """specs: list of (kernel_size, stride) from the input onward. -> int"""
    pass  # TODO

In [ ]:
grade('nb09', 'receptive_field', receptive_field, verbose=False)

In [ ]:
rf = use('nb09', 'receptive_field', receptive_field)

print("three stacked 3x3, stride 1 :", rf([(3, 1)] * 3), " (same as one 7x7)")
print("one 7x7, stride 1           :", rf([(7, 1)]))
print("  params for C=64: 3x3 stack %d vs 7x7 %d"
      % (3 * (3 * 3 * 64 * 64), 7 * 7 * 64 * 64))
print()
print("conv-pool-conv-pool-conv    :", rf([(3, 1), (2, 2), (3, 1), (2, 2), (3, 1)]))
print("ResNet stem 7x7s2 + 3x3s2   :", rf([(7, 2), (3, 2)]))

depths = np.arange(1, 21)
rfs = [rf([(3, 1)] * int(d)) for d in depths]
rfs_pool = [rf(sum([[(3, 1), (2, 2)] for _ in range(int(d) // 2)], [])) for d in depths]
fig, ax = plt.subplots(figsize=(6.2, 3.4))
plot_series(ax, depths, [("3x3 convs only", rfs),
                         ("3x3 conv + 2x2 pool", rfs_pool)],
            "Receptive field growth", "number of 3x3 conv layers",
            "receptive field (pixels)", direct_labels=False)
ax.axhline(32, color='#8a8887', linestyle='--', linewidth=0.9)
ax.annotate("CIFAR image = 32px", xy=(1, 34), fontsize=8, color='#52514e')
plt.show()

Stride-1 convs grow the receptive field **linearly** (2 pixels per layer);
adding stride or pooling makes it grow **exponentially**. That is the real
reason for downsampling — not just to save compute, but so that deep units can
see the whole image at all. Without it you would need 16 layers just to cover a
32×32 CIFAR image.

## 3. The architectures

| Year | Model | Idea | Top-5 |
|---|---|---|---|
| 1998 | **LeNet-5** | conv + pool + FC, trained by backprop | — |
| 2012 | **AlexNet** | LeNet at scale: ReLU, dropout, GPUs | 16.4% |
| 2014 | **VGG-16** | only 3×3 convs, uniform and deep | 7.3% |
| 2014 | **GoogLeNet** | parallel branches, 1×1 bottlenecks, no FC head | 6.7% |
| 2015 | **ResNet-152** | skip connections; 152 layers | **3.6%** |

Three ideas carry most of the weight:

**1×1 convolutions** (GoogLeNet). A 1×1 conv has no spatial extent, so it is
purely a learned linear mix *across channels* — a cheap way to cut 256 channels
to 64 before an expensive 5×5. This is the "bottleneck" in every modern block.

**Parallel branches** (Inception). Rather than choosing a filter size, apply
1×1, 3×3, 5×5 and pooling simultaneously and concatenate. Let the network
weight them.

**Skip connections** (ResNet). Below.

In [ ]:
from solutions.nb09 import (make_lenet, make_vgg_block, InceptionBlock,
                            SmallResNet, PlainNet, count_parameters as cpm)

def count_parameters(module, trainable_only=True):
    """Total number of scalars across module.parameters(). One line."""
    pass  # TODO

In [ ]:
grade('nb09', 'count_parameters', count_parameters, verbose=False)

In [ ]:
cnt = use('nb09', 'count_parameters', count_parameters)

x = torch.randn(2, 3, 32, 32)
for nm, m in [('LeNet-5', make_lenet()),
              ('VGG block 3->64 x2', make_vgg_block(3, 64, 2)),
              ('Inception block', InceptionBlock(192, 64, 96, 128, 16, 32, 32)),
              ('SmallResNet w=32', SmallResNet(width=32))]:
    try:
        shp = tuple(m(x).shape) if 'Inception' not in nm else \
              tuple(m(torch.randn(2, 192, 8, 8)).shape)
    except Exception:
        shp = '-'
    print("{:<22} {:>9,} params   out {}".format(nm, cnt(m), shp))

### The 1×1 bottleneck, quantified

In [ ]:
# 256 channels -> 5x5 conv -> 64 channels, on a 28x28 map.
direct = cpar(256, 64, 5)
bottle = cpar(256, 32, 1) + cpar(32, 64, 5)
print("direct   256 --5x5--> 64        : {:>9,} params".format(direct))
print("bottleneck 256 -1x1-> 32 -5x5-> 64: {:>9,} params".format(bottle))
print("%.1fx fewer, and it adds a nonlinearity" % (direct / bottle))

## 4. TODO 3 — the residual block

The problem ResNet solved was not overfitting. It was that a **56-layer plain
network had higher *training* error than a 20-layer one** — it could not even
optimize. Depth was making optimization harder, not the model weaker.

The fix:

$$ y = \mathcal{F}(x) + x $$

Learn the *residual* $\mathcal{F}(x) = y - x$ rather than $y$ directly. Two
consequences:

- **The identity is free.** If the best thing a block can do is nothing, it
  just drives $\mathcal{F} \to 0$, which is easy. Extra depth can no longer
  hurt.
- **The gradient has a highway.** Since $\frac{\partial y}{\partial x} =
  \frac{\partial \mathcal{F}}{\partial x} + 1$, that `+1` guarantees gradient
  reaches earlier layers *no matter how small* $\partial\mathcal{F}/\partial x$
  becomes. It is the add gate from notebook 04 — a distributor — doing exactly
  what it always does.

Two details in the implementation:

- **Add before the final ReLU**, not after.
- When the shape changes (stride ≠ 1, or channel count differs) the shortcut
  cannot be the identity — project it with a 1×1 conv.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        # bias=False because BatchNorm immediately removes any constant
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)

        self.downsample = None
        if stride != 1 or in_ch != out_ch:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch))

    def forward(self, x):
        # TODO
        #   identity = x, or self.downsample(x) when it is not None
        #   out = relu(bn1(conv1(x)))
        #   out = bn2(conv2(out))
        #   out = out + identity      <- BEFORE the final relu
        #   return relu(out)
        pass

In [ ]:
# Check it yourself: shapes, and the gradient-highway property.
from solutions.nb09 import ResidualBlock as RefBlock

def check_block(cls):
    try:
        b = cls(32, 32)
        o = b(torch.randn(2, 32, 16, 16))
        assert tuple(o.shape) == (2, 32, 16, 16)
        b2 = cls(32, 64, 2)
        o2 = b2(torch.randn(2, 32, 16, 16))
        assert tuple(o2.shape) == (2, 64, 8, 8)
        return True
    except Exception as e:
        print("  not working yet (%s)" % type(e).__name__)
        return False

ok = check_block(ResidualBlock)
Block = ResidualBlock if ok else RefBlock
print("using your ResidualBlock" if ok else "using the reference ResidualBlock")

b = Block(32, 32)
inp = torch.randn(2, 32, 16, 16)
print("\nstride 1, same channels :", tuple(b(inp).shape), "(identity shortcut)")
print("stride 2, 32 -> 64      :", tuple(Block(32, 64, 2)(inp).shape),
      "(projected shortcut)")

In [ ]:
# The gradient highway, measured.
#
# Measure ATTENUATION: inject a gradient at the output and see what fraction
# of its norm survives to the input. BatchNorm is deliberately left out here
# and the init is deliberately under-scaled (std 0.05 where He would be 0.118),
# because BatchNorm already prevents vanishing on its own -- and the point is
# to isolate what the skip connection alone does.
class DemoBlock(nn.Module):
    def __init__(self, c, skip, scale):
        super().__init__()
        self.c1 = nn.Conv2d(c, c, 3, padding=1)
        self.c2 = nn.Conv2d(c, c, 3, padding=1)
        self.skip = skip
        for m in (self.c1, self.c2):
            nn.init.normal_(m.weight, 0.0, scale)
            nn.init.zeros_(m.bias)

    def forward(self, x):
        o = F.relu(self.c1(x))
        o = self.c2(o)
        if self.skip:
            o = o + x
        return F.relu(o)

def attenuation(depth, skip, scale=0.05, seed=0):
    torch.manual_seed(seed)
    net = nn.Sequential(*[DemoBlock(16, skip, scale) for _ in range(depth)])
    xg = torch.randn(4, 16, 8, 8, requires_grad=True)
    out = net(xg)
    g = torch.randn_like(out)
    out.backward(g)
    return (xg.grad.norm() / g.norm()).item()

print("fraction of the output gradient's norm that survives to the input:\n")
print("  depth | with skips | without skips")
for d in [2, 4, 8, 16, 32]:
    s_, p_ = attenuation(d, True), attenuation(d, False)
    print("   %3d  |  %.4f    |   %.3e" % (d, s_, p_))

**With skips, the surviving fraction is flat in depth** — around 0.7 whether
the stack is 2 blocks or 32. Without them it falls off a cliff and has
underflowed to exactly zero by depth 32. That is the `+1` in
$\partial y/\partial x = \partial \mathcal{F}/\partial x + 1$, doing its job.

Two honest caveats:

- This isolates the skip connection by removing BatchNorm and under-scaling
  the init. **BatchNorm also prevents this kind of vanishing**, so in a modern
  net you are never choosing between them — you use both.
- The historical evidence for ResNet was not a gradient-norm plot. It was that
  a 56-layer plain net had **higher training error** than a 20-layer one *even
  with* BatchNorm, and that adding skips removed the effect. Vanishing
  gradients are part of the story, not all of it.

## 5. Train a real convnet

Everything so far, applied. CIFAR-10, a small ResNet, on CPU.

In [ ]:
data = get_cifar10(num_train=20000, num_val=2000, num_test=2000,
                   flatten=False, subtract_mean=True)
pixel_std = data['X_train'].std()

def to_t(a):
    return torch.tensor(a / pixel_std, dtype=torch.float32)

X_tr, X_va, X_te = to_t(data['X_train']), to_t(data['X_val']), to_t(data['X_test'])
y_tr = torch.tensor(data['y_train'])
y_va = torch.tensor(data['y_val'])
y_te = torch.tensor(data['y_test'])
print("train", tuple(X_tr.shape), " val", tuple(X_va.shape))

In [ ]:
from solutions.nb09 import train_torch, evaluate

net = SmallResNet(width=32)
print("SmallResNet: {:,} parameters\n".format(cnt(net)))

t0 = time.time()
hist = train_torch(net, X_tr, y_tr, X_va, y_va, epochs=10, batch_size=128,
                   lr=2e-3, augment=True, verbose=True)
print("\n%.0f s" % (time.time() - t0))
print("TEST accuracy %.3f" % evaluate(net, X_te, y_te))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.5))
axes[0].plot(np.arange(1, len(hist['loss']) + 1), hist['loss'], color=SERIES[0])
axes[0].set_title("Training loss"); axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss"); axes[0].set_axisbelow(True)
plot_series(axes[1], np.arange(1, len(hist['train_acc']) + 1),
            [('train', hist['train_acc']), ('val', hist['val_acc'])],
            "Accuracy", "epoch", "accuracy")
plt.show()

In [ ]:
# The control: identical run with augmentation switched off.
net_noaug = SmallResNet(width=32)
hist_noaug = train_torch(net_noaug, X_tr, y_tr, X_va, y_va, epochs=10,
                         batch_size=128, lr=2e-3, augment=False, verbose=False)

print("%-16s %8s %8s %8s" % ("", "train", "val", "gap"))
for nm, h in [("no augment", hist_noaug), ("augment", hist)]:
    print("%-16s %8.3f %8.3f %8.3f"
          % (nm, h['train_acc'][-1], h['val_acc'][-1],
             h['train_acc'][-1] - h['val_acc'][-1]))

fig, ax = plt.subplots(figsize=(6.6, 3.6))
ep = np.arange(1, len(hist['val_acc']) + 1)
plot_series(ax, ep, [('train, no aug', hist_noaug['train_acc']),
                     ('val,   no aug', hist_noaug['val_acc']),
                     ('train, aug',    hist['train_acc']),
                     ('val,   aug',    hist['val_acc'])],
            "Data augmentation: random flip + random crop", "epoch",
            "accuracy", direct_labels=False)
plt.show()

### Read this one carefully too

At a 10-epoch budget, augmentation barely moves **validation accuracy** — the
two runs land within about a point of each other. What it moves decisively is
the **gap**: roughly 0.17 without augmentation versus 0.06 with it. The
augmented model is nowhere near memorising its training set.

So at this budget augmentation is not buying accuracy, it is buying
**headroom** — and the headroom is what pays. Re-running the same comparison
with `epochs=25` gives:

| | train | val | gap |
|---|---|---|---|
| no augmentation | **1.000** | 0.746 | 0.254 |
| augmentation | 0.887 | **0.824** | 0.063 |

The un-augmented model hits 100% training accuracy — it has memorised all
20,000 images and has nothing left to learn from, so its validation accuracy
is pinned at ~0.75 no matter how much longer you train. The augmented one is
still improving and ends nearly **8 points higher**. (Change `epochs=10` to
`25` above to reproduce this; it takes a few minutes.)

The general lesson, and it applies to every regularizer: **a technique that
closes the train/val gap without immediately raising validation accuracy has
not failed — it has moved the bottleneck.** Judge a regularizer at the budget
you actually intend to train at, not at a short one.

### The whole series, in one table

| Notebook | Model | CIFAR-10 accuracy |
|---|---|---|
| 01 | k-NN on raw pixels | ~0.29 |
| 02–03 | linear classifier (softmax) | ~0.39 |
| 05 | 2-layer fully-connected net | ~0.46 |
| 09 | small ResNet + augmentation | **~0.74** (0.82 at 25 epochs) |

Same data, same loss function, same optimizer family. Everything that changed
is **architecture** — the priors you build into the model about what images
are.

In [ ]:
# What the first conv layer learned. Compare with notebooks 02 and 05.
w = net.stem[0].weight.detach().numpy()     # (32, 3, 3, 3)
fig, axes = plt.subplots(2, 16, figsize=(12, 1.8))
for i, ax in enumerate(axes.ravel()):
    f = w[i].transpose(1, 2, 0)
    f = (f - f.min()) / max(np.ptp(f), 1e-9)
    ax.imshow(f); ax.axis('off')
fig.suptitle("Learned 3x3 filters, first conv layer", fontsize=10, x=0.01, ha='left')
plt.show()

Nine pixels each, so do not expect textbook Gabor stripes — at 3×3 there is
barely room for one. What you can read off them is **colour opponency**: most
filters put warm against cool, or bright against dark, across the kernel. Those
are colour and intensity *contrast* detectors, which is exactly what a first
layer should learn and what the hand-designed Sobel and sharpen kernels in
notebook 06 also did.

The honest comparison with notebook 05's fully-connected weights is not "these
are prettier". It is that there are only `32 × 3 × 3 × 3 = 864` numbers here,
each receiving gradient from every spatial position of every image — versus
614,400 weights in notebook 05, each seeing one fixed position. That ratio is
the whole argument for convolution.

In [ ]:
# Where it still fails.
net.eval()
with torch.no_grad():
    preds = net(X_te[:500]).argmax(1)
wrong = (preds != y_te[:500]).nonzero().flatten()[:20]
lbl = ["%s->%s" % (CLASSES[y_te[i]], CLASSES[preds[i]]) for i in wrong]
show_images((X_te[wrong] * pixel_std).numpy(), labels=np.arange(len(wrong)),
            class_names=lbl, ncols=10, title="Misclassified (true -> predicted)",
            scale=1.2)
plt.show()

from collections import Counter
conf = Counter(("%s as %s" % (CLASSES[y_te[i]], CLASSES[preds[i]]))
               for i in (preds != y_te[:500]).nonzero().flatten())
print("most common confusions:")
for k, v in conf.most_common(6):
    print("  %-22s %d" % (k, v))

The confusions are almost all *semantically plausible* — cat/dog, car/truck,
deer/horse. Contrast this with notebook 01, where kNN's nearest neighbours were
matched on background colour. The model has learned something about objects.

## 6. Where to go next

- **Transfer learning.** You will almost never train from scratch. Take a
  network pretrained on ImageNet, replace the final layer, and fine-tune.
  With a small dataset, freeze everything but the last layer; with more data,
  fine-tune the whole thing at a low learning rate (~1/10th). Your existing
  `15_transfer_learning.ipynb` and `03_transfer_learning_detection.ipynb`
  cover the practice.
- **CS231n Lectures 10–16:** RNNs and attention, detection and segmentation,
  visualizing what convnets learn, generative models, self-supervision.
- **Modern additions** post-dating the course: Vision Transformers, CLIP,
  diffusion models, SAM.

## 7. Recap

1. Name the three levels of PyTorch abstraction and when to use each.
2. Why does PyTorch accumulate gradients instead of overwriting them, and what
   is the symptom of forgetting `zero_grad()`?
3. What exactly does `model.eval()` change?
4. Why must you not put a softmax before `nn.CrossEntropyLoss`?
5. Which layers dominate parameters? Which dominate FLOPs? Why the difference?
6. What does global average pooling replace, and what does it save?
7. Why do three 3×3 convs beat one 7×7?
8. Write the receptive-field recurrence. Why is downsampling necessary for
   deep units to see the whole image?
9. What does a 1×1 convolution compute, and why is it called a bottleneck?
10. What specific failure did ResNet fix, and why is that failure surprising?
11. Why does `y = F(x) + x` help gradients reach early layers?
12. Why `bias=False` on a conv followed by BatchNorm?

In [ ]:
# show_solution('nb09', 'receptive_field')
# show_solution('nb09', 'ResidualBlock')